# Three-dimensional nearly incompressible elasticity

GaLS and Taylor–Hood local spaces retain six rigid motions and impose the physical compressibility identity. The exact family keeps its force bounded as lambda tends to infinity.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.gals3d import solve_elasticity_gals_3d
from pymhm.tetrahedral import TetraMesh

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


In [ ]:
def displacement(x):
    """Affine solenoidal displacement with nonzero boundary data."""
    return np.column_stack((1+x[:,0], 2-x[:,1], x[:,0]+x[:,1]))

with threadpool_limits(1):
    patch = solve_elasticity_gals_3d(TetraMesh.unit_cube(), degree=2,
                                   lame_lambda=np.inf, dirichlet=displacement,
                                   mean_pressure=2.)
    errors = {"displacement": patch.l2_error(displacement),
              "pressure": patch.pressure_l2_error(2.),
              "compressibility": patch.compressibility_l2()}
assert max(errors.values()) < 1e-10
errors


Pressure means are prescribed only at exact incompressibility on a full displacement boundary. Finite lambda fixes the weighted pressure integral through the displacement boundary flux. Cauchy traction is symmetric; it differs from the grad-grad flow pseudotraction.

In [ ]:
folder = ROOT / "examples/results/gals3d"
report = json.loads((folder / "campaign.json").read_text())
rows = report["rows"]
assert len(rows) == 36
for row in rows:
    assert hashlib.sha256((folder / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
    assert row["max_error_quadrature_relative_difference"] < 1e-8
    assert row["backward_residual"] < 1e-8
for case in ["gals-p1", "gals-p2", "th-p2"]:
    refined = [row for row in rows if row["case"] == case and row["study"] == "refinement"]
    assert len(refined) == 5
    assert refined[-1]["displacement_l2"] < refined[0]["displacement_l2"]
    assert refined[-1]["pressure_l2"] < refined[0]["pressure_l2"]
[(row["case"], row["lame_lambda"], row["displacement_l2"], row["pressure_l2"])
 for row in rows if row["study"] == "lambda"]

primal = json.loads((ROOT / "examples/results/gals3d/primal/campaign.json").read_text())
assert len(primal["primal_control"]) == 4
assert all(row["local_refinement_precision"] == "extended" for row in primal["primal_control"])


In [ ]:
figures = ROOT / "docs/figures/gals3d"
for name in ["refinement", "lambda", "gals-p1-fields", "gals-p2-fields", "th-p2-fields"]:
    display(Image(filename=str(figures / f"{name}.png")))


The refinement and parameter studies are manufactured 3D validation, not reproductions of the paper's two-dimensional figures. The primal comparison uses the same operator, material and boundary data, with finite lambda. These finite tests do not establish a uniform inf-sup bound for every mesh. Raw stresses are not an H(div) reconstruction.